# 04 · Assemble a leak-free backtest dataset

> Goal: join point-in-time factor features to forward-return labels — a clean
> (X, y) frame you can train any model on.

## What you need

- **An API key** with PRO+ tier (the `/labels/{ticker}` endpoint is PRO+ gated).
  Without one, this notebook explains the pattern but can't execute the
  labels fetch.
- `pandas` for the join.

In [1]:
# Setup — works with or without an API key.
# With FACTORWEAVE_API_KEY set, we use the full API (10,000+ tickers).
# Without one, we fall back to /demo/{ticker} (AAPL, MSFT, NVDA, AMZN, GOOGL, META, TSLA, JPM).
import os, json
import requests

API_BASE = "https://factorweave.com/api"
API_KEY = os.environ.get("FACTORWEAVE_API_KEY")
DEMO_TICKERS = ["AAPL", "MSFT", "NVDA", "AMZN", "GOOGL", "META", "TSLA", "JPM"]
MODE = "live" if API_KEY else "demo"
print(f"Running in {MODE} mode.", "Key prefix:", (API_KEY[:8] + '…') if API_KEY else "(none)")


def fw_demo(ticker: str) -> dict:
    """Demo endpoint — no auth, 8 sample tickers, current snapshot only."""
    r = requests.get(f"{API_BASE}/demo/{ticker}", timeout=10)
    r.raise_for_status()
    return r.json()


def fw_features(ticker: str, **kwargs) -> dict:
    """Authed features endpoint when a key is available; demo fallback otherwise."""
    if API_KEY:
        r = requests.get(f"{API_BASE}/features/{ticker}",
                         params=kwargs,
                         headers={"X-API-Key": API_KEY},
                         timeout=10)
        r.raise_for_status()
        return r.json()
    # Demo fallback — reshape demo response to look like the authed one
    d = fw_demo(ticker)
    return {"rows": [{"ticker": d["ticker"], "date": d["as_of"], **d["factors"]}]}


def fw_top(factor: str, n: int = 25) -> dict:
    """Top-N by a factor. Needs auth for the full universe; in demo mode we
    rank the 8 demo tickers locally."""
    if API_KEY:
        r = requests.get(f"{API_BASE}/top",
                         params={"factor": factor, "n": n},
                         headers={"X-API-Key": API_KEY},
                         timeout=10)
        r.raise_for_status()
        return r.json()
    # Demo fallback — fetch each demo ticker, sort locally
    rows = []
    for t in DEMO_TICKERS:
        d = fw_demo(t)
        if factor in d["factors"]:
            rows.append({"ticker": t, "date": d["as_of"], factor: d["factors"][factor]})
    rows.sort(key=lambda r: r[factor], reverse=True)
    return {"rows": rows[:n]}


def fw_similar(ticker: str, method: str = "cosine", limit: int = 10) -> dict:
    """Similarity search. Demo endpoint includes pre-computed `similar` set."""
    if API_KEY:
        r = requests.get(f"{API_BASE}/vector-search/similar/{ticker}",
                         params={"method": method, "limit": limit, "min_lookback_days": 30},
                         headers={"X-API-Key": API_KEY},
                         timeout=10)
        r.raise_for_status()
        return r.json()
    # Demo fallback — uses the `similar` array baked into the demo response
    d = fw_demo(ticker)
    return {"ticker": ticker, "method": "cosine (demo)", "neighbors": d.get("similar", [])[:limit]}


def fw_market_context() -> dict:
    """Universe analytics. Public on FREE, fuller on HOBBY+."""
    headers = {"X-API-Key": API_KEY} if API_KEY else {}
    r = requests.get(f"{API_BASE}/market-context", params={"latest": 1}, headers=headers, timeout=10)
    if r.status_code == 401:
        return {"_note": "market-context requires auth in demo mode"}
    r.raise_for_status()
    return r.json()


Running in demo mode. Key prefix: (none)


## The pattern (always)

In [2]:
import pandas as pd
import requests

def fetch_features_history(ticker, start, end):
    if not API_KEY:
        return None
    r = requests.get(f"{API_BASE}/features/{ticker}",
                     params={"start_date": start, "end_date": end},
                     headers={"X-API-Key": API_KEY}, timeout=20)
    r.raise_for_status()
    j = r.json()
    return pd.DataFrame(j.get("rows", j) if isinstance(j, dict) else j)


def fetch_labels(ticker, start, end):
    if not API_KEY:
        return None
    r = requests.get(f"{API_BASE}/labels/{ticker}",
                     params={"start_date": start, "end_date": end},
                     headers={"X-API-Key": API_KEY}, timeout=20)
    if r.status_code == 403:
        print(f"[labels] {ticker}: tier-gated — PRO+ required")
        return None
    r.raise_for_status()
    j = r.json()
    return pd.DataFrame(j.get("labels", j) if isinstance(j, dict) else j)


TICKER, START, END = "AAPL", "2024-01-01", "2024-12-31"
X = fetch_features_history(TICKER, START, END)
y = fetch_labels(TICKER, START, END)

if X is None or y is None:
    print("This step needs an API key — set FACTORWEAVE_API_KEY and re-run.")
    print("The shape of what you'd get back:")
    print("   X: ~252 rows × 28 factor columns, indexed by date")
    print("   y: ~252 rows × 6 label columns (fwd_ret_1d/5d/20d + binary targets)")
else:
    Xi = X.set_index("date")
    yi = y.set_index("date")["fwd_ret_20d"]
    df = Xi.join(yi).dropna(subset=["fwd_ret_20d"])
    print(f"Joined frame: {df.shape[0]} rows × {df.shape[1]} cols")
    print()
    print(df.head())

This step needs an API key — set FACTORWEAVE_API_KEY and re-run.
The shape of what you'd get back:
   X: ~252 rows × 28 factor columns, indexed by date
   y: ~252 rows × 6 label columns (fwd_ret_1d/5d/20d + binary targets)


## What's leak-free about it

Each `as_of` date carries factors computed only from data through that day,
joined to a forward return computed from *future* prices (T+1 onward).
Train on factors_t, predict fwd_ret(t→t+h). No bleed.

## Where to go from here

- Cross-sectional model: stack many tickers, regress `fwd_ret_20d` on
  factor columns at the same `as_of`. Train on early dates, test on
  later — walk-forward.
- Time-series model per ticker: ARIMA-style on the labels themselves
  (rarely beats unconditional mean for individual names, but worth a check).
- Classifier: predict `sign(fwd_ret_20d)` — useful sanity check.

> ⚠ Honest expectation: our own leak-free walk-forward testing across
> the full universe and methods (cosine similarity, label-aware,
> supervised PLS, gradient boosting) shows cross-sectional IC ≈ 0. The
> data is for **research**, not for return prediction. Use the dataset
> to explore, refute, and assemble — not to claim a signal.

## Next

→ `05-regime-conditioning.ipynb` — split any of these analyses by the SPY
volatility regime, since factor behavior differs by regime.